# LAB 3: Creating a simple AI Agent with Langchain

## Initiation

In [12]:
# Run this cell if you have not installed langchain in your environment
!pip install langchain
!pip install langchain-openai


[notice] A new release of pip is available: 24.2 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 24.2 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [13]:
# loading environment variables 
from dotenv import load_dotenv
load_dotenv(override=True)  # take environment variables

True

In [ ]:
# Explain how to generate apikey from Tavily dashboard
TAVILY_API_KEY = "<insert your tavily key here>"

In [14]:
# Initiating Langchain Chat Models
from langchain.chat_models import init_chat_model
model = init_chat_model("gpt-4.1-mini", model_provider= "openai")

In [15]:
# Since we need a tool from langchain community, we need to install the package first
!pip install langchain-community


[notice] A new release of pip is available: 24.2 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Tool Calling with Langchain

In [11]:
# Tool 1: Google Search
from langchain_community.tools.tavily_search import TavilySearchResults
search = TavilySearchResults(max_results=2)
search.invoke("langchain tutorial")

[{'title': 'LangChain Crash Course For Beginners - YouTube',
  'url': 'https://www.youtube.com/watch?v=nAmC7SoVLd8',
  'content': '# LangChain Crash Course For Beginners | LangChain Tutorial\n\ncodebasics\n10397 likes\n426851 views\n30 Jun 2023\nLangChain is an open-source framework that allows you to build applications using LLMs (Large Language Models). In this crash course for LangChain, we are going to cover the following topics, [...] 00:00 Introduction\n00:22 What is Langchain?\n04:38 Langchain installation and setup\n06:28 LLMs, Prompt Templates\n10:12 Chains\n11:57 Simple Sequential Chain\n15:30 Sequential Chain\n18:00 Build Streamlit App \n27:40 Agents\n37:24 Memory\n\nLink to the source code: \n\nLangchain tutorial playlist: \n\nDo you want to learn technology from me? Check  for my affordable video courses.\n\nNeed help building software or data analytics/AI solutions? My company  can help. Click on the Contact button on that website.',
  'score': 0.83093774},
 {'title': 'Bu

In [8]:
# Tool 2: Weather
import requests
from langchain_core.tools import tool

@tool(parse_docstring=True)
def get_weather(latitude, longitude):
    """
        Use this tool to get current temperature and wind speed around a certain location. Search longitude and latitude from the internet.

        args:
            latitude : latitude value of the location
            longitude : longitude value of the location
    """
    response = requests.get(f"https://api.open-meteo.com/v1/forecast?latitude={latitude}&longitude={longitude}&current=temperature_2m,wind_speed_10m&hourly=temperature_2m,relative_humidity_2m,wind_speed_10m")
    data = response.json()
    return data['current']

In [9]:
# Testing Tool 2
get_weather.invoke({'latitude': "-6.8999", "longitude": "108.89998"})

{'time': '2025-07-17T08:45',
 'interval': 900,
 'temperature_2m': 33.1,
 'wind_speed_10m': 6.1}

In [17]:
!pip install yfinance

^C


     ---------------------------------------- 0.0/949.2 kB ? eta -:--:--
     ---------------------------------------- 0.0/949.2 kB ? eta -:--:--
     ---------------------------------------- 0.0/949.2 kB ? eta -:--:--
     ---------------------------------------- 0.0/949.2 kB ? eta -:--:--
     ----------- ---------------------------- 262.1/949.2 kB ? eta -:--:--
     ----------- ---------------------------- 262.1/949.2 kB ? eta -:--:--
     ----------- ---------------------------- 262.1/949.2 kB ? eta -:--:--
     ----------- ---------------------------- 262.1/949.2 kB ? eta -:--:--
     ------------------- ---------------- 524.3/949.2 kB 328.9 kB/s eta 0:00:02
     ------------------- ---------------- 524.3/949.2 kB 328.9 kB/s eta 0:00:02
     ----------------------------- ------ 786.4/949.2 kB 435.8 kB/s eta 0:00:01
     ----------------------------- ------ 786.4/949.2 kB 435.8 kB/s eta 0:00:01
     ----------------------------- ------ 786.4/949.2 kB 435.8 kB/s eta 0:00:01
     ---


[notice] A new release of pip is available: 24.2 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [18]:
# Tool 3: Stock Price 
import yfinance as yf

@tool(parse_docstring=True)
def get_stock_price(ticker: str) -> str:
    '''
        This function is called to get the latest closing stock price of a company. 
        
        Args:
            ticker (str): The stock ticker symbol (e.g., 'AAPL', 'GOOG').

        Returns:
            str: A message with the latest closing stock price, or an error message if data is unavailable.
    '''
    stock = yf.Ticker(ticker)
    data = stock.history(period="1d")
    if data.empty:
        return f"No data found for {ticker}."
    latest_price = data['Close'].iloc[-1]
    return f"The latest closing price of {ticker} is ${latest_price:.2f}"

In [19]:
get_stock_price.invoke("AAPL")

'The latest closing price of AAPL is $210.16'

In [20]:
tools = [search, get_weather, get_stock_price]

## Building The AI Agent with Langchain

In [21]:
model_with_tools = model.bind_tools(tools)

In [22]:
# No Tool Calling
from langchain_core.messages import HumanMessage

response = model_with_tools.invoke([HumanMessage(content="Hi!")])

print(f"ContentString: {response.content}")
print(f"ToolCalls: {response.tool_calls}")

ContentString: Hello! How can I assist you today?
ToolCalls: []


In [23]:
# calling weather
response = model_with_tools.invoke([HumanMessage(content="What's the weather in SF?")])

print(f"ContentString: {response.content}")
print(f"ToolCalls: {response.tool_calls}")

ContentString: 
ToolCalls: [{'name': 'tavily_search_results_json', 'args': {'query': 'San Francisco weather'}, 'id': 'call_1tGCosaJuMUKm0eT5bdcbWXi', 'type': 'tool_call'}]


In [24]:
from langchain.agents import create_tool_calling_agent
from langchain import hub

# Get the prompt to use - you can modify this!
prompt = hub.pull("hwchase17/openai-functions-agent")
agent = create_tool_calling_agent(model, tools, prompt)

In [25]:
prompt.pretty_print()

================================ System Message ================================

You are a helpful assistant

============================= Messages Placeholder =============================

{chat_history}

================================ Human Message =================================

{input}

============================= Messages Placeholder =============================

{agent_scratchpad}


In [26]:
from langchain.agents import AgentExecutor

agent_executor = AgentExecutor(agent=agent, tools=tools)

In [27]:
agent_executor.invoke({"input": "whats the weather in sf?"})

{'input': 'whats the weather in sf?',
 'output': 'The current weather in San Francisco is 13.9°C with a wind speed of 8.6 m/s.'}

In [28]:
agent_executor.invoke({"input": "Give me the stock price of Tesla"})

{'input': 'Give me the stock price of Tesla',
 'output': 'The latest closing stock price of Tesla (TSLA) is $321.67.'}

In [29]:
# Calling two tools
for event in agent_executor.stream({"input": "what's the wind speed in Sumber?"}):
    print(event)

{'actions': [ToolAgentAction(tool='get_weather', tool_input={'latitude': -7.4358, 'longitude': 111.2105}, log="\nInvoking: `get_weather` with `{'latitude': -7.4358, 'longitude': 111.2105}`\n\n\n", message_log=[AIMessageChunk(content='', additional_kwargs={'tool_calls': [{'index': 0, 'id': 'call_3pF9f4UqN9XrQiqiZVrJwikw', 'function': {'arguments': '{"latitude":-7.4358,"longitude":111.2105}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'finish_reason': 'tool_calls', 'model_name': 'gpt-4.1-mini-2025-04-14', 'service_tier': 'default'}, id='run--5ca8f2c0-c23d-4b98-89a5-a20bdb2ed579', tool_calls=[{'name': 'get_weather', 'args': {'latitude': -7.4358, 'longitude': 111.2105}, 'id': 'call_3pF9f4UqN9XrQiqiZVrJwikw', 'type': 'tool_call'}], tool_call_chunks=[{'name': 'get_weather', 'args': '{"latitude":-7.4358,"longitude":111.2105}', 'id': 'call_3pF9f4UqN9XrQiqiZVrJwikw', 'index': 0, 'type': 'tool_call_chunk'}])], tool_call_id='call_3pF9f4UqN9XrQiqiZVrJwikw')], 'messages': [AI

## END